In [0]:
import subprocess

CONTROL_SOCKET = "/tmp/databricks-ssh"
SSH_HOST = "databricks@pub.worldb.dedyn.io"
SSH_PORT = 8889
SSH_KEY = "/Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free"

ssh_base = (
    f"ssh "
    f"-S {CONTROL_SOCKET} "
    f"-p {SSH_PORT} "
    f"-i {SSH_KEY} "
    f"-o BatchMode=yes "
    f"{SSH_HOST}"
)


def start_ssh_tunnel() -> None:
    command = (
        f"ssh "
        f"-M "
        f"-S {CONTROL_SOCKET} "
        f"-o ControlPersist=yes "
        f"-o ServerAliveInterval=30 "
        f"-o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile=/tmp/known_hosts "
        f"-o BatchMode=yes "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-L 127.0.0.1:9092:kafka-4:9092 "
        f"-fNT "
        f"{SSH_HOST}"
    )

    subprocess.run(command, shell=True, check=True)

def check_ssh_tunnel() -> bool:
    result = subprocess.run(
        f"{ssh_base} -O check",
        shell=True,
        text=True,
        capture_output=True,
    )

    print(result.stdout or result.stderr)
    return result.returncode == 0


def stop_ssh_tunnel() -> None:
    subprocess.run(
        f"{ssh_base} -O exit",
        shell=True,
        check=False,
    )

In [0]:
start_ssh_tunnel()

In [0]:
!ps -ef | grep ssh

In [0]:
check_ssh_tunnel()

In [0]:
stop_ssh_tunnel()

In [0]:
!ps -ef | grep ssh